# MemoryGate walkthrough

This notebook walks through the reusable Strands Decider 2B pattern in this folder, one decision at a time. The decider's answers are **replayed** from `quickstart_answers.json`, which was recorded from a real Strands Decider 2B server. Everything else (the policy, the curator's bands, the prompt block) runs live, so you can edit a cell and rerun it with no server, GPU or AWS account.

To ask a live decider instead, replace `replay_ask(...)` with `http_ask("http://127.0.0.1:8099")`.

In [1]:
from memory_gate import (COST, INSTRUCTIONS, OPTIONS, MemoryCurator, MemoryGate,
                         render_context, replay_ask)
from quickstart import MEMORIES, NEAR_NEWS, NEWS, SCENES

ask = replay_ask("quickstart_answers.json")
gate = MemoryGate(ask)
for m in MEMORIES:
    print(m["id"], m["text"])

m1 The user has been vegetarian since 2019.
m2 The user's dog Biscuit passed away last month.
m3 The user used to walk Biscuit every morning by the lake.
m4 The user broke up with Meera during a trip to Paris last year.
m5 The user prefers aisle seats on flights.
m6 The user lives in Pune and works as a data analyst.


## 1. The question

Each memory becomes one typed `choice` question. The option descriptions carry most of the signal, so they're written as carefully as the question. All memories go in **one** request: the decider reads the conversation once, and each memory only adds its own tokens.

In [2]:
print(INSTRUCTIONS.format(memory=MEMORIES[1]["text"]))
print()
for label, description in OPTIONS.items():
    print(f"{label:8} {description}")

An AI assistant is about to reply to the user's latest message. It remembers this fact about the user:
The user's dog Biscuit passed away last month.
How should the assistant treat this fact in its reply?

mention  use it openly: it makes the reply more useful or personal (a preference, constraint, plan or interest that fits the request), and bringing it up feels natural
quiet    use it silently: it is related to the request, but saying it out loud could hurt, embarrass or upset the user, so it should only make the reply more careful
skip     ignore it: it has no bearing on what a good reply would say


## 2. A harmless question near a painful memory

The user asks about getting a puppy. Their dog died last month.

In [3]:
def show(result):
    for v in result.verdicts:
        p = "  ".join(f"{k}={x:.2f}" for k, x in v.probabilities.items())
        print(f"{v.action:8} {p}   {v.text}")
        if v.action != v.label:
            print(f"         ↳ {v.reason}")

title, conversation = SCENES[1]
print("user:", conversation[-1][1], "\n")
puppy = gate.run(conversation, MEMORIES)
show(puppy)

user: I'm thinking of getting a puppy. Thoughts? 

skip     mention=0.06  quiet=0.09  skip=0.85   The user has been vegetarian since 2019.
quiet    mention=0.20  quiet=0.26  skip=0.53   The user's dog Biscuit passed away last month.
         ↳ leaned skip (p=0.53), but quiet has the lower expected cost (0.42 vs 0.57)
skip     mention=0.09  quiet=0.06  skip=0.86   The user used to walk Biscuit every morning by the lake.
skip     mention=0.03  quiet=0.12  skip=0.86   The user broke up with Meera during a trip to Paris last year.
skip     mention=0.02  quiet=0.05  skip=0.93   The user prefers aisle seats on flights.
skip     mention=0.11  quiet=0.06  skip=0.83   The user lives in Pune and works as a data analyst.


The decider's top label for Biscuit is *skip*, but it isn't sure. The policy doesn't just take the top label: it picks the action with the lowest **expected cost** under the probabilities. Here is that arithmetic for the Biscuit memory:

In [4]:
biscuit = next(v for v in puppy.verdicts if "Biscuit passed" in v.text)
p = biscuit.probabilities
for action, row in COST.items():
    terms = " + ".join(f"{p[t]:.2f}×{c}" for t, c in row.items())
    print(f"{action:8} {terms} = {sum(p[t] * c for t, c in row.items()):.2f}")
print("\nchosen:", biscuit.action)

mention  0.20×0.0 + 0.26×3.0 + 0.53×0.5 = 1.06
quiet    0.20×1.0 + 0.26×0.0 + 0.53×0.4 = 0.42
skip     0.20×1.5 + 0.26×1.0 + 0.53×0.0 = 0.57

chosen: quiet


So the memory reaches the LLM as *sensitive background, never mention*: the reply can suggest waiting a little before a new puppy without bringing up Biscuit.

## 3. The same memory, when the user opens the door

In [5]:
title, conversation = SCENES[2]
print("user:", conversation[-1][1], "\n")
show(gate.run(conversation, MEMORIES))

user: It's been a month since Biscuit died and I still can't walk past the lake. How do I cope? 

skip     mention=0.03  quiet=0.11  skip=0.86   The user has been vegetarian since 2019.
mention  mention=0.66  quiet=0.20  skip=0.14   The user's dog Biscuit passed away last month.
mention  mention=0.71  quiet=0.14  skip=0.16   The user used to walk Biscuit every morning by the lake.
skip     mention=0.03  quiet=0.13  skip=0.84   The user broke up with Meera during a trip to Paris last year.
skip     mention=0.02  quiet=0.09  skip=0.90   The user prefers aisle seats on flights.
skip     mention=0.05  quiet=0.13  skip=0.83   The user lives in Pune and works as a data analyst.


Now ignoring Biscuit would be cold, and the decider moves both Biscuit memories to *mention*. A static "sensitive" tag can't do this: it would keep them unsaid in both conversations.

## 4. The cost table is yours to change

Replay the puppy scene on the **same** decider answers, changing only how much a painful mention costs (`COST["mention"]["quiet"]`, 3 by default):

In [6]:
for c in (0.5, 1, 2, 3, 5):
    cost = {a: dict(row) for a, row in COST.items()}
    cost["mention"]["quiet"] = c
    v = next(v for v in MemoryGate(ask, cost=cost).run(SCENES[1][1], MEMORIES).verdicts
             if "Biscuit passed" in v.text)
    print(f"painful mention costs {c:>3}:  {v.action:8} {v.text}")

painful mention costs 0.5:  mention  The user's dog Biscuit passed away last month.
painful mention costs   1:  quiet    The user's dog Biscuit passed away last month.
painful mention costs   2:  quiet    The user's dog Biscuit passed away last month.
painful mention costs   3:  quiet    The user's dog Biscuit passed away last month.
painful mention costs   5:  quiet    The user's dog Biscuit passed away last month.


Make a painful mention cheap and the agent brings up the dead dog when you ask about a puppy. Taking the decider's top label as-is would *skip* the memory entirely, so the reply couldn't even be careful. The expected-cost policy lands in between: use it, don't say it.

No retraining, no new prompt: the trade-off lives in a table your team can read and argue about.

## 5. What the LLM actually receives

In [7]:
title, conversation = SCENES[3]
print("user:", conversation[-1][1], "\n")
print(render_context(gate.run(conversation, MEMORIES)))

user: Can you suggest a Thai dish I could cook tonight? 

Things you remember about the user that you MAY bring up if useful:
- The user has been vegetarian since 2019.


## 6. The write side: news makes old memories outdated

`MemoryCurator` checks the stored memories closest to the news, in one request: outdated, repeat, or unaffected, plus a yes/no *is this a past event?* that keeps finished events from being retired.

In [8]:
print("user:", NEWS[0][1], "\n")
nearby = [m for m in MEMORIES if m["id"] in NEAR_NEWS]
curation = MemoryCurator(ask).run(NEWS, nearby)
for u in curation.updates:
    print(f"{u.action:7} P(outdated)={u.probabilities['outdated']:.2f}  "
          f"P(past event)={u.p_past:.2f}   {u.text}")
print("\nstore the new fact:", curation.store_new)

user: Big news: I just moved to Bangalore for a new job at a bank! 

keep    P(outdated)=0.23  P(past event)=0.75   The user's dog Biscuit passed away last month.
keep    P(outdated)=0.29  P(past event)=0.45   The user prefers aisle seats on flights.
retire  P(outdated)=0.91  P(past event)=0.40   The user lives in Pune and works as a data analyst.

store the new fact: True


## 7. Wiring it into a Strands agent

The gate runs in a Strands `InterventionHandler.before_invocation`, once per user message and before any model call. It returns a `Transform` that rewrites the system prompt, so the LLM only ever sees what the gate allowed. The full version, with remember / forget / tone riding in the same request, is `src/tact/agent.py`.

```python
class MemoryGateIntervention(InterventionHandler):
    def before_invocation(self, event, **kwargs):
        result = self.gate.safe_run(conversation, self.store.candidates(latest, k=8))
        prompt = f"{SYSTEM_PROMPT}\n\n{render_context(result)}"
        return Transform(apply=lambda e: setattr(e.agent, "system_prompt", prompt))

agent = Agent(model=BedrockModel(model_id="us.amazon.nova-lite-v1:0"),
              interventions=[MemoryGateIntervention(gate, store)])
```

`safe_run` means a decider outage gives the agent **no** memories for that turn, never a stale or ungated block.